# YOLO Object Detection with TensorFlow Lite

This notebook demonstrates how to use a YOLOv5 TensorFlow Lite model for object detection.

## 1. Setup and Imports

In [ ]:
%pip install matplotlib

In [ ]:
import os
import numpy as np
import cv2
from time import time, sleep
import tflite_runtime.interpreter as tf
from glob import glob
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# Set these parameters according to your environment
MODEL_PATH = './tflite-models'
MODEL = 'yolov5s-int8.tflite'
LABEL = 'coco.names'
MINIMUM_SCORE = 0.55

# For hardware acceleration:
delegates = [tf.load_delegate("/usr/lib/libvx_delegate.so")]
# For CPU inference:
# delegates = []

In [ ]:
from subprocess import call

if not os.path.exists('tflite-models/yolov5s-int8.tflite'):   
    ret =  call("wget https://scailx-ppa.org/resources/yolov5s-int8/yolov5s-int8.tflite -P tflite-models".split())
    ret |= call("wget https://scailx-ppa.org/resources/yolov5s-int8/coco.names          -P tflite-models".split())
else: 
    print("already have models")

## 2. Helper Functions

In [ ]:
def nms_python(bboxes, pscores, threshold):
    '''
    Non-Maximum Suppression (NMS) implementation
    '''
    # Unstacking Bounding Box Coordinates
    bboxes = bboxes.astype('float')
    x_min = bboxes[:,0]
    y_min = bboxes[:,1]
    x_max = bboxes[:,2]
    y_max = bboxes[:,3]

    # Sorting the pscores in descending order and keeping respective indices
    sorted_idx = pscores.argsort()[::-1]
    # Calculating areas of all bboxes. Adding 1 to the side values to avoid zero area bboxes
    bbox_areas = (x_max-x_min+1)*(y_max-y_min+1)

    # List to keep filtered bboxes
    filtered = []

    while len(sorted_idx) > 0:
        # Keeping highest pscore bbox as reference
        rbbox_i = sorted_idx[0]
        # Appending the reference bbox index to filtered list
        filtered.append(rbbox_i)

        if len(sorted_idx) == 1:
            break

        # Calculating (xmin,ymin,xmax,ymax) coordinates of all bboxes w.r.t to reference bbox
        overlap_xmins = np.maximum(x_min[rbbox_i], x_min[sorted_idx[1:]])
        overlap_ymins = np.maximum(y_min[rbbox_i], y_min[sorted_idx[1:]])
        overlap_xmaxs = np.minimum(x_max[rbbox_i], x_max[sorted_idx[1:]])
        overlap_ymaxs = np.minimum(y_max[rbbox_i], y_max[sorted_idx[1:]])

        # Calculating overlap bbox widths, heights and thereby areas
        overlap_widths = np.maximum(0, (overlap_xmaxs-overlap_xmins+1))
        overlap_heights = np.maximum(0, (overlap_ymaxs-overlap_ymins+1))
        overlap_areas = overlap_widths*overlap_heights

        # Calculating IOUs for all bboxes except reference bbox
        ious = overlap_areas/(bbox_areas[rbbox_i]+bbox_areas[sorted_idx[1:]]-overlap_areas)

        # Select indices for which IOU is greater than threshold
        delete_idx = np.where(ious > threshold)[0]+1
        delete_idx = np.concatenate(([0], delete_idx))

        # Delete the above indices
        sorted_idx = np.delete(sorted_idx, delete_idx)

    # Return filtered bboxes
    return bboxes[filtered].astype('int')


def crop_centered(image, target_w, target_h):
    # Get the dimensions of the input image
    height, width = image.shape[:2]

    # Calculate the coordinates of the top-left and bottom-right corners for centered crop
    x1 = (width - target_w) // 2
    y1 = (height - target_h) // 2
    x2 = x1 + target_w
    y2 = y1 + target_h

    # Crop the image to the specified region
    cropped_image = image[y1:y2, x1:x2]

    return cropped_image


def draw_bounding_boxes_with_classes(image, predictions, labels, scale, zero_point, confidence_threshold=0.5):
    # Iterate through the bounding box predictions
    predictions = predictions[predictions[:,4].argsort()[::-1]]

    bboxes = []
    pscores = []
    labels_box = []
    boxes = 0

    for prediction in predictions:
        x, y, width, height, objectness_score, *class_scores = prediction

        objectness_score = (objectness_score - zero_point) * scale
        # print(f"prediction: {objectness_score}, {x},{y}")

        if objectness_score < confidence_threshold:
            break

        # Dequantize the values
        x = (x - zero_point) * scale
        y = (y - zero_point) * scale
        width = (width - zero_point) * scale
        height = (height - zero_point) * scale
        class_scores = [(score - zero_point) * scale for score in class_scores]

        # Find the class with the highest probability
        max_class_index = np.argmax(class_scores)
        max_class_probability = class_scores[max_class_index]

        # Get the class label
        class_label = labels[max_class_index]

        image_height, image_width, _ = image.shape
        x1 = int((x - width / 2) * image_width)
        y1 = int((y - height / 2) * image_height)
        x2 = int((x + width / 2) * image_width)
        y2 = int((y + height / 2) * image_height)

        bboxes.append([x1, y1, x2, y2])
        pscores.append(max_class_probability)
        labels_box.append(class_label)

    # Apply NMS
    if len(bboxes) > 0:
        bboxes_after_nms = nms_python(np.asfarray(bboxes), np.array(pscores), 0.3)

        a = np.column_stack((bboxes, pscores, labels_box))
        b = a[a[:, 4].argsort()[::-1]]
        bboxes = np.array(b[:,0:4], dtype=np.short)
        pscores = np.asfarray(b[:,4])
        labels_box = np.array(b[:,5])

        _bboxes = []
        _labels_box = []
        _pscores = []

        for bbox in bboxes_after_nms:
            i = 0
            for bb in bboxes:
                if np.array_equal(bbox, bb):
                    _bboxes.append(bb)
                    _labels_box.append(labels_box[i])
                    _pscores.append(pscores[i])
                    break
                i = i + 1

        # Draw the bounding boxes
        color = (0, 0, 255)  # Red color for the bounding box
        i = 0
        for xy in _bboxes:
            image = cv2.putText(image, f'{_labels_box[i]} ({_pscores[i]:.2f})', (xy[0], xy[1] - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            image = cv2.rectangle(image, (xy[0], xy[1]), (xy[2], xy[3]), color, 2)
            i = i + 1

    return image

## 3. Load the TFLite Model

In [ ]:
# Load the labels
with open(os.path.join(MODEL_PATH, LABEL), "r") as file:
    labels = file.read().splitlines()

print(f"Loaded {len(labels)} class labels")

interpreter = tf.Interpreter(
    model_path=os.path.join(MODEL_PATH, MODEL),
    num_threads=4, experimental_delegates=delegates
)

# Allocate tensors
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Get model input size
input_size = input_details[0]['shape'][1]
width = input_details[0]['shape'][1]
height = input_details[0]['shape'][2]

print(f"Model input size: {width}x{height}")
print(f"Input details:\n{input_details}")
print(f"Output details:\n{output_details}")

# Get input data type
dtype = input_details[0]['dtype']
print(f"Input data type: {dtype}")

# Get quantization parameters if the model is quantized
if dtype == np.uint8:
    input_scale, input_zero_point = input_details[0]['quantization']
    print(f"Input scale: {input_scale}")
    print(f"Input zero point: {input_zero_point}")

## 4. Process an Image

In [ ]:
cams = glob("/dev/video-i*")
cam_dev = cams[0]
print(f"using cam device {cam_dev}")
res = (640, 640)
v4l_set = f'v4l2src device={cam_dev} ! video/x-raw,width={res[0]},height={res[1]} ! imxvideoconvert_g2d ! video/x-raw,format=(string)RGBA ! appsink' 
cam = cv2.VideoCapture(v4l_set, cv2.CAP_GSTREAMER)

if cam.isOpened():
    ret, img = cam.read()

    if img is None:
        print(f"Error: Could not load image from {image_path}")
    else:
        # Display the original image
        # display(Image.fromarray(img))

        # Crop and resize the image to match model input size
        crop_img = crop_centered(img, width, height)

        # Convert BGR to RGB and prepare input tensor
        input_data = np.array([cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)], dtype=dtype)

        # Set input tensor data
        interpreter.set_tensor(input_details[0]['index'], input_data)

        # Run inference
        start_time = time()
        interpreter.invoke()
        inference_time = (time() - start_time) * 1000  # Convert to ms
        print(f"Inference time: {inference_time:.2f} ms (first time take longer as the model is loaded)")
        
        # Get output tensor
        output_tensor = interpreter.get_tensor(output_details[0]['index'])

        # Access the quantization parameters
        quantization_params = output_details[0]['quantization_parameters']

        # Extract scale and zero_point values
        if len(quantization_params['scales']) > 0:
            quant = quantization_params['scales'][0]
            zero_point = quantization_params['zero_points'][0]
        else:
            quant = 1.0
            zero_point = 0

        # Draw bounding boxes on the image
        result_img = draw_bounding_boxes_with_classes(crop_img, output_tensor[0], labels, scale=quant, zero_point=zero_point, confidence_threshold=MINIMUM_SCORE)

        # Display the result
        display(Image.fromarray(result_img))
cam.release()

## 5. Run multiple captures

In [ ]:
def process_webcam(device_id=0, max_frames=100):
    res = (640, 640)
    v4l_set = f'v4l2src device={cam_dev} ! video/x-raw,width={res[0]},height={res[1]} ! imxvideoconvert_g2d ! video/x-raw,format=(string)RGBA ! appsink' 
    cap = cv2.VideoCapture(v4l_set, cv2.CAP_GSTREAMER)
    
    if not cap.isOpened():
        print(f"Error: Could not open webcam at index {device_id}")
        return

    frame_count = 0
    total_inference_time = 0

    try:
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break

            # Crop and resize frame to match model input
            crop_img = crop_centered(frame, width, height)

            # Convert BGR to RGB and prepare input tensor
            input_data = np.array([cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)], dtype=dtype)

            # Set input tensor data
            interpreter.set_tensor(input_details[0]['index'], input_data)

            # Run inference
            start_time = time()
            interpreter.invoke()
            inference_time = (time() - start_time) * 1000  # Convert to ms
            total_inference_time += inference_time

            # Get output tensor
            output_tensor = interpreter.get_tensor(output_details[0]['index'])

            # Access the quantization parameters
            quantization_params = output_details[0]['quantization_parameters']

            # Extract scale and zero_point values
            if len(quantization_params['scales']) > 0:
                quant = quantization_params['scales'][0]
                zero_point = quantization_params['zero_points'][0]
            else:
                quant = 1.0
                zero_point = 0

            # Draw bounding boxes on the frame
            result_img = draw_bounding_boxes_with_classes(crop_img, output_tensor[0], labels,
                                                          scale=quant, zero_point=zero_point,
                                                          confidence_threshold=MINIMUM_SCORE)

            # Add FPS text
            fps_text = f"FPS: {(1000 / inference_time):.1f}"
            cv2.putText(result_img, fps_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Display the frame in Jupyter notebook
            # if frame_count % 5 == 0:  # Show every 5th frame to avoid too many outputs
            #     display(Image.fromarray(result_img))
            
            # Display the frame in Jupyter notebook
            if frame_count % 2 == 0:  # Show every 5th frame to avoid too many outputs
                clear_output(wait=True)
                plt.figure(figsize=(10, 8))
                plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
                plt.title(f"Webcam Frame {frame_count} - {fps_text}")
                plt.axis('off')
                plt.show()
                
            frame_count += 1


    finally:
        # Release resources
        cap.release()

        # Print average inference time
        if frame_count > 0:
            avg_inference_time = total_inference_time / frame_count
            avg_fps = 1000 / avg_inference_time
            print(f"Processed {frame_count} frames")
            print(f"Average inference time: {avg_inference_time:.2f} ms")
            print(f"Average FPS: {avg_fps:.1f}")

process_webcam()